# Phase 6 - Seven-model Java clone demo
Select a **T4 GPU**. Upload the supplied `EmbedEd_Phase6_source.zip` when prompted. This is a private, temporary notebook webpage, not public hosting.
Seed 13 for C0-C6; fixed HF revisions and exact saved thresholds. No training or threshold tuning. The two editable examples are illustrative and unlabeled. Scores are not probabilities.
Java is text only; never compiled or executed. The app does not log or save user snippets. Do not paste proprietary code. Preparation downloads about **3.51 GB**, including six checkpoints and the untuned base. No HF secret is needed for these public assets. Never paste credentials into source or outputs.


In [ ]:
from pathlib import Path
import hashlib, json, zipfile, subprocess, sys, threading
from google.colab import files
ROOT = Path('/content/EmbedEd_phase6')
uploaded = files.upload()
candidates = [n for n in uploaded if n.endswith('.zip')]
if len(candidates) != 1: raise ValueError('Upload exactly the Phase 6 source ZIP.')
with zipfile.ZipFile(candidates[0]) as archive:
    manifest = json.loads(archive.read('phase6_source_manifest.json'))
    if set(archive.namelist()) != set(manifest) | {'phase6_source_manifest.json'}: raise ValueError('Unexpected archive contents')
    for name, digest in manifest.items():
        rel = Path(name)
        if rel.is_absolute() or '..' in rel.parts: raise ValueError('Unsafe archive path')
        data = archive.read(name)
        if hashlib.sha256(data).hexdigest() != digest: raise ValueError('Source hash mismatch')
        target = ROOT / name
        if target.exists() and target.read_bytes() != data: raise ValueError('Existing source differs; choose a fresh ROOT directory')
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.46.3', 'rank-bm25==0.2.2', 'pytest', 'matplotlib'], check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Select a T4 GPU runtime first.')
print('Source verified; accelerator:', torch.cuda.get_device_name())


## Prepare fixed assets
This cell verifies sizes, LFS SHA-256 / Git blob hashes, and an exact allowlist. Reruns reuse the Hub cache. A changed identity is rejected instead of overwriting unrelated assets.


In [ ]:
ASSETS = Path('/content/phase6_assets')
subprocess.run([sys.executable, '-m', 'demo.prepare_compare', '--dest', str(ASSETS)], cwd=ROOT, check=True)


## Verify before serving
Checks 14 real scores/decisions against the existing inference path, exercises HTTP error handling, and measures cold latency / peak GPU memory. The long-input check is synthetic text for truncation testing, not a benchmark. Only the checked-in examples and metadata are saved in the evidence.


In [ ]:
EVIDENCE = Path('/content/phase6_evidence')
subprocess.run([sys.executable, '-m', 'demo.verify_compare', '--assets', str(ASSETS), '--out', str(EVIDENCE), '--device', 'cuda'], cwd=ROOT, check=True)
proof = json.loads((EVIDENCE / 'live_verification.json').read_text())
assert proof['status'] == 'passed' and proof['parity_scores_checked'] == 14
print('Parity passed; cold comparison:', proof['live_result']['latency_ms'], 'ms; peak GPU:', proof['live_result']['peak_cuda_mb'], 'MB')


## Open the private webpage
The app binds to loopback and opens through Colab's notebook port proxy. No public tunnel is created. Code entered in the page goes to this runtime for inference, not a third-party model API. One comparison runs at a time. If the runtime disconnects, restart these cells; no placeholder predictions are substituted.


In [ ]:
from demo.compare import ComparisonEngine, make_server
from google.colab import output
if globals().get('phase6_server') is not None:
    phase6_server.shutdown()
    phase6_server.server_close()
phase6_engine = ComparisonEngine(ASSETS, device='cuda')
phase6_server = make_server(phase6_engine, port=7860)
phase6_worker = threading.Thread(target=phase6_server.serve_forever, daemon=True)
phase6_worker.start()
output.serve_kernel_port_as_iframe(7860, height=1800, cache_in_notebook=False)


## Stop when finished (run separately)
Stop the app below. Download verification evidence if wanted, then use **Runtime → Disconnect and delete runtime** to release Colab compute. The verification evidence contains no user snippets; it does not capture later comparisons. Do not leave a T4 running unattended.


In [ ]:
# Deliberately manual: leave this cell unrun while using the webpage.
# phase6_server.shutdown(); phase6_server.server_close()
# phase6_server = None; phase6_engine = None
# import gc; gc.collect(); torch.cuda.empty_cache()
# files.download(str(EVIDENCE / 'live_verification.json'))
print('When finished, run the shutdown lines, then disconnect and delete this runtime.')
